In [2]:
import requests
import cloudscraper
from curl_cffi import requests as stealth_requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import math
import re
from datetime import datetime
import urllib.robotparser
import json
from pathlib import Path

# 1. Directory Management
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "Notebooks" else current_dir
output_dir = project_root / "Data" / "Raw"
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Target Categories
categories = {
    "Laptop": "https://www.teknosa.com/laptop-notebook-c-116004",
    "Monitor": "https://www.teknosa.com/monitor-c-116006",
    "Washing Machine": "https://www.teknosa.com/camasir-makinesi-c-10302",
    "A/C Units": "https://www.teknosa.com/klima-c-117007002",
    "Vacuum Cleaner": "https://www.teknosa.com/elektrikli-supurge-c-117005",
    "Robot Vacuum": "https://www.teknosa.com/robot-supurge-c-117005003",
    "Headphones": "https://www.teknosa.com/kulaklik-c-100006",
    "Television": "https://www.teknosa.com/televizyon-c-101001",
    "Dishwasher": "https://www.teknosa.com/bulasik-makinesi-c-10303",
    "Refrigerator": "https://www.teknosa.com/buzdolabi-c-10301"
}

# 3. Request Configurations & Verbose Cascade Fetcher
request_headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Accept-Language": "tr-TR,tr;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8"
}

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

def fetch_url_robust_verbose(url, timeout=15):
    """Verbose Triple-Cascade Fetcher reporting every network attempt out loud."""
    print("      -> [Tier 1] Standard Requests...", end=" ")
    try:
        response = requests.get(url, headers=request_headers, timeout=timeout)
        if response.status_code not in [401, 403, 503]:
            print(f"SUCCESS (Status {response.status_code})")
            return response
        print(f"BLOCKED (Status {response.status_code})")
    except requests.exceptions.RequestException as e:
        print(f"FAILED ({e})")

    print("      -> [Tier 2] Cloudscraper Engine...", end=" ")
    try:
        response = scraper.get(url, timeout=timeout)
        if response.status_code not in [401, 403, 503]:
            print(f"SUCCESS (Status {response.status_code})")
            return response
        print(f"BLOCKED (Status {response.status_code})")
    except Exception as e:
        print(f"FAILED ({e})")

    print("      -> [Tier 3] Stealth curl_cffi...", end=" ")
    try:
        response = stealth_requests.get(url, impersonate="chrome", timeout=timeout)
        if response.status_code == 200:
            print("SUCCESS (Status 200)")
        else:
            print(f"FAILED (Status {response.status_code})")
        return response
    except Exception as e:
        print(f"CRITICAL ERROR ({e})")
        return None

# 4. Compliance Engine
robots_text = """User-agent: *
Disallow: /sepet
Disallow: /checkout
Disallow: /hesabim
Disallow: /product-compare*
Disallow: /kurumsal/*
Disallow: /urun-soru-cevap
Disallow: /karsilastir/*
Disallow: /arama/*
Disallow: /login*
Disallow: /*gad_source
Disallow: /*?shopId=
Disallow: /inform/gowit
Disallow: /*?q=
Disallow: /*gowitAdId=*
Disallow: *s=*
Disallow: *text=*
Disallow: *sort=*
Disallow: *chatbotai=*
"""

print("Initializing Stealth Compliance Engine...")
rp = urllib.robotparser.RobotFileParser()
rp.set_url("https://www.teknosa.com/robots.txt")

try:
    print("  -> Fetching robots.txt rules...")
    robots_response = fetch_url_robust_verbose("https://www.teknosa.com/robots.txt", timeout=15)
    if robots_response and robots_response.status_code == 200:
        rp.parse(robots_response.text.splitlines())
    else:
        raise ValueError("Non-200 Status")
except Exception:
    print("  -> Live rules failed. Applying offline bypass rules.")
    rp.parse(robots_text.splitlines())

# 5. Hybrid Extraction Function
def extract_products_from_page(soup_obj, cat_name, base_url):
    extracted_products = []
    
    # Strategy A: JSON-LD Schema
    script_tags = soup_obj.find_all("script", type="application/ld+json")
    for tag in script_tags:
        try:
            data = json.loads(tag.text)
            if data.get("@type") == "ItemList" and "itemListElement" in data:
                for item in data["itemListElement"]:
                    prod = item.get("item", {})
                    if prod.get("@type") == "Product":
                        url_string = prod.get("url", "")
                        sku = url_string.split("-p-")[-1] if "-p-" in url_string else "Unknown SKU"
                        
                        extracted_products.append({
                            "Product_Name": prod.get("name", "Unknown Name").strip(),
                            "Brand": prod.get("brand", "Unknown Brand").strip(),
                            "Retailer_SKU": sku.strip(),
                            "Category": cat_name,
                            "Source_URL": "https://www.teknosa.com" + url_string if url_string.startswith("/") else url_string,
                            "Extraction_Method": "JSON-LD"
                        })
        except json.JSONDecodeError:
            continue

    # Strategy B: Fallback HTML DOM
    if not extracted_products:
        product_cards = soup_obj.find_all("div", attrs={"data-product-id": True})
        for card in product_cards:
            try:
                if not card.get("data-product-name"):
                    continue
                name = card.get("data-product-name", "Unknown Name").strip()
                brand = card.get("data-product-brand", "Unknown Brand").strip()
                sku = card.get("data-product-id", "Unknown SKU").strip()
                url_suffix = card.get("data-product-url", "").strip()
                full_url = "https://www.teknosa.com" + url_suffix if url_suffix else base_url
                
                extracted_products.append({
                    "Product_Name": name,
                    "Brand": brand,
                    "Retailer_SKU": sku,
                    "Category": cat_name,
                    "Source_URL": full_url,
                    "Extraction_Method": "HTML_DOM"
                })
            except Exception:
                continue

    return extracted_products

# 6. Main Scraping Loop with Stratified Random Sampling
portfolio_data = []

for category_name, base_url in categories.items():
    print(f"\n========================================")
    print(f"Evaluating category: {category_name}...")
    print(f"========================================")
    
    if not rp.can_fetch("ErguvanESG_CaseStudy_Bot", base_url):
        print(f"  -> BLOCKED by robots.txt: Skipping {base_url}")
        continue
        
    print(f"  -> Executing Reconnaissance on Page 1...")
    try:
        recon_response = fetch_url_robust_verbose(base_url, timeout=15)
        if not recon_response:
            continue
            
        recon_soup = BeautifulSoup(recon_response.content, "html.parser")
        
        total_pages = 1
        info_div = recon_soup.find("div", class_="plp-info")
        
        if info_div:
            cleaned_text = info_div.text.replace('.', '').replace(',', '')
            match = re.search(r'(\d+)', cleaned_text)
            if match:
                total_results = int(match.group(1))
                total_pages = max(1, math.ceil(total_results / 20))
                print(f"  -> Recon Data: Found {total_results} total results. Calculated {total_pages} total pages.")
        
        # Randomly select up to 7 pages from the available range
        pages_to_scrape = random.sample(range(1, total_pages + 1), min(7, total_pages))
        pages_to_scrape.sort() 
        print(f"  -> Randomly selected pages to scrape: {pages_to_scrape}\n")
        
        for page_num in pages_to_scrape:
            print(f"    -> Extracting Page {page_num}...")
            
            if page_num == 1:
                category_products = extract_products_from_page(recon_soup, category_name, base_url)
                
                # Verbose Extraction Logging
                json_count = sum(1 for p in category_products if p["Extraction_Method"] == "JSON-LD")
                html_count = len(category_products) - json_count
                
                portfolio_data.extend(category_products)
                print(f"        -> Extracted {len(category_products)} items [JSON-LD: {json_count} | HTML DOM: {html_count}] (Cached from recon).")
            else:
                time.sleep(random.uniform(3.0, 5.0))
                target_url = f"{base_url}?page={page_num}"
                response = fetch_url_robust_verbose(target_url, timeout=15)
                
                if response and response.status_code == 200:
                    page_soup = BeautifulSoup(response.content, "html.parser")
                    category_products = extract_products_from_page(page_soup, category_name, target_url)
                    
                    # Verbose Extraction Logging
                    json_count = sum(1 for p in category_products if p["Extraction_Method"] == "JSON-LD")
                    html_count = len(category_products) - json_count
                    
                    portfolio_data.extend(category_products)
                    print(f"        -> Extracted {len(category_products)} items [JSON-LD: {json_count} | HTML DOM: {html_count}].")
                else:
                    print(f"        -> Failed to extract page {page_num}.")
                    
    except Exception as e:
        print(f"  -> Category error occurred: {e}")

# 7. Data Export
if portfolio_data:
    df = pd.DataFrame(portfolio_data)
    df = df.drop_duplicates(subset=["Retailer_SKU"])
    
    date_str = datetime.today().strftime('%Y%m%d')
    file_name = f"teknosa_portfolio_snapshot_{date_str}.csv"
    file_path = output_dir / file_name
    
    df.to_csv(file_path, index=False)
    print(f"\nPhase 1 Complete! Successfully saved {len(df)} unique products to:")
    print(f" -> {file_path}")
else:
    print("\nPhase 1 Failed: No products were scraped.")

Initializing Stealth Compliance Engine...
  -> Fetching robots.txt rules...
      -> [Tier 1] Standard Requests... BLOCKED (Status 403)
      -> [Tier 2] Cloudscraper Engine... BLOCKED (Status 403)
      -> [Tier 3] Stealth curl_cffi... SUCCESS (Status 200)

Evaluating category: Laptop...
  -> Executing Reconnaissance on Page 1...
      -> [Tier 1] Standard Requests... BLOCKED (Status 403)
      -> [Tier 2] Cloudscraper Engine... BLOCKED (Status 403)
      -> [Tier 3] Stealth curl_cffi... SUCCESS (Status 200)
  -> Recon Data: Found 30480 total results. Calculated 1524 total pages.
  -> Randomly selected pages to scrape: [40, 105, 309, 576, 989, 1255, 1404]

    -> Extracting Page 40...
      -> [Tier 1] Standard Requests... BLOCKED (Status 403)
      -> [Tier 2] Cloudscraper Engine... BLOCKED (Status 403)
      -> [Tier 3] Stealth curl_cffi... SUCCESS (Status 200)
        -> Extracted 20 items [JSON-LD: 20 | HTML DOM: 0].
    -> Extracting Page 105...
      -> [Tier 1] Standard Request